# Test MIST Glioma on a real DICOM case

This notebook runs the full `mist-glioma` pipeline on a GPU: DICOM reading, N4 and registration, HD-BET, and five-fold MIST inference. It uses public case UPENN-GBM-00622 and the published BraTS 2025 weights.

**Before you start**

1. Choose **Runtime > Change runtime type > GPU**.
2. Choose **Runtime > Run all**.

The notebook installs into its own environment, so it does not change Colab's packages or need a runtime restart. The last cell prints a summary you can paste back.

The example data is from the [UPENN-GBM collection](https://doi.org/10.7937/TCIA.709X-DN49) (CC BY 4.0, citation required). By running this notebook you agree to the TCIA Data Usage Policy. **The output is for research testing only and is not for clinical use.**

## Settings

In [ ]:
SOURCE = "github"       # "github", or "upload" to test a local tar file of the repository.
GITHUB_REF = "main"     # Branch or tag to use when SOURCE is "github".
RUN_NIFTI_COMPARISON = True   # Run again from NIfTI and compare with the DICOM result.
RUN_POSTPROCESS = True        # Run again with --postprocess.

ROOT = "/content"

## Helpers

In [ ]:
import os
import shutil
import subprocess
import sys
import tarfile
import time
from pathlib import Path

ROOT_PATH = Path(ROOT)
SRC = ROOT_PATH / "mist-glioma"
VENV = ROOT_PATH / "venv"
PY = str(VENV / "bin" / "python")
CASE = ROOT_PATH / "data" / "UPENN-GBM-00622"
NIFTI = ROOT_PATH / "nifti"
OUT = ROOT_PATH / "output"
SEQUENCES = ("t1", "t2", "tc", "fl")
RESULTS = {}


def run(*command, check=True, cwd=None):
    """Run a command in the test environment, streaming and returning its output."""
    env = dict(os.environ)
    env["PATH"] = f"{VENV / 'bin'}:{env['PATH']}"
    for name in ("MPLBACKEND", "PYTHONPATH"):
        env.pop(name, None)
    process = subprocess.Popen(
        [str(c) for c in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, errors="replace", env=env, cwd=cwd,
    )
    lines = []
    for line in process.stdout:
        print(line, end="")
        lines.append(line)
    code = process.wait()
    output = "".join(lines)
    if check and code != 0:
        raise RuntimeError(f"Exit code {code}: {' '.join(str(c) for c in command)}")
    return code, output


def segment(output_name, inputs, *extra):
    """Run mist_glioma and record how long it took."""
    start = time.time()
    run(
        "mist_glioma",
        "--t1", inputs["t1"], "--t2", inputs["t2"], "--tc", inputs["tc"], "--fl", inputs["fl"],
        "--output", OUT / output_name, "--device", "cuda", *extra,
    )
    minutes = (time.time() - start) / 60
    RESULTS[f"{output_name}_minutes"] = round(minutes, 1)
    print(f"{output_name}: finished in {minutes:.1f} minutes")

## 1. Get the code

In [ ]:
if SRC.exists():
    print(f"Using existing {SRC}")
elif SOURCE == "upload":
    from google.colab import files
    uploaded = files.upload()
    name = next(n for n in uploaded if n.endswith(".tar.gz"))
    with tarfile.open(name) as archive:
        archive.extractall(ROOT_PATH, filter="data")
elif SOURCE == "github":
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_REF,
         "https://github.com/mist-medical/mist-glioma.git", str(SRC)],
        check=True,
    )
else:
    raise ValueError(f"Unknown SOURCE: {SOURCE}")

assert (SRC / "mist_glioma" / "image_io.py").exists(), "This code has no DICOM support."

## 2. Install

This takes a few minutes, mostly for PyTorch.

In [ ]:
start = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(["uv", "venv", str(VENV), "--python", "3.12", "--clear"], check=True)
subprocess.run(["uv", "pip", "install", "-q", "--python", PY, f"{SRC}[test]"], check=True)
RESULTS["install_minutes"] = round((time.time() - start) / 60, 1)
print(f"Installed in {RESULTS['install_minutes']} minutes")

_, versions = run(PY, "-c", '''
import importlib.metadata as m, torch
for p in ("mist-medical", "torch", "monai", "HD-BET", "antspyx", "SimpleITK", "huggingface-hub"):
    print(p, m.version(p))
print("cuda", torch.cuda.is_available())
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
''')
RESULTS["versions"] = dict(line.rsplit(" ", 1) if " " in line and not line.startswith("gpu") else line.split(" ", 1)
                           for line in versions.strip().splitlines() if " " in line)
assert RESULTS["versions"].get("cuda") == "True", "No GPU found. Choose Runtime > Change runtime type > GPU."

## 3. Run the unit tests

In [ ]:
code, _ = run(PY, "-m", "pytest", "-q", "-p", "no:cacheprovider", check=False, cwd=SRC)
RESULTS["unit_tests_passed"] = code == 0

## 4. Download the example case and the weights

This also confirms that the weights download without a Hugging Face login.

In [ ]:
run(PY, SRC / "examples" / "download_upenn_gbm_00622.py", "--output", CASE)
run(PY, "-c", '''
from mist_glioma.model_assets import get_model_directories
print(get_model_directories()["brats2025"])
''', cwd=ROOT_PATH)
RESULTS["weights_downloaded"] = True

## 5. Run the full pipeline from DICOM

This is the main test. It is also the first real run of the N4 fix.

In [ ]:
DICOM_INPUTS = {name: CASE / name for name in SEQUENCES}
segment("dicom", DICOM_INPUTS)

## 6. Convert the inputs to NIfTI

The viewers below use these copies, and step 9 uses them as inputs.

In [ ]:
NIFTI.mkdir(exist_ok=True)
run(PY, "-c", f'''
import ants
from mist_glioma.image_io import read_image
for name in {SEQUENCES!r}:
    ants.image_write(read_image("{CASE}/" + name), "{NIFTI}/" + name + ".nii.gz")
''', cwd=ROOT_PATH)

## 7. Check the DICOM result

These checks catch broken outputs, not poor accuracy. This case has no ground truth in this space, so also look at the overlays in step 8.

In [ ]:
import nibabel as nib
import numpy as np
from nibabel.processing import resample_from_to
from scipy import ndimage

LABELS = {1: "NETC", 2: "SNFH", 3: "ET", 4: "RC"}
REGIONS = {**{name: [label] for label, name in LABELS.items()}, "TC": [1, 3], "WT": [1, 2, 3]}


def load_mask(run_name, file_name="tumor_rc_mask.nii.gz"):
    image = nib.load(OUT / run_name / file_name)
    return image, np.rint(np.asanyarray(image.dataobj)).astype(np.uint8)


def volumes_cc(mask, image):
    voxel_cc = float(np.prod(image.header.get_zooms()[:3])) / 1000
    return {name: round(float(np.isin(mask, labels).sum()) * voxel_cc, 1) for name, labels in REGIONS.items()}


reference = nib.load(NIFTI / "tc.nii.gz")
tumor_image, tumor = load_mask("dicom")
brain_image, brain = load_mask("dicom", "brain_mask.nii.gz")
brain_cc = float((brain > 0).sum()) * float(np.prod(brain_image.header.get_zooms()[:3])) / 1000
tumor_voxels = np.isin(tumor, list(LABELS))

checks = {
    "tumor mask is on the T1 post-contrast grid": tumor.shape == reference.shape and np.allclose(tumor_image.affine, reference.affine, atol=1e-2),
    "brain mask is on the T1 post-contrast grid": brain.shape == reference.shape and np.allclose(brain_image.affine, reference.affine, atol=1e-2),
    "labels are all in 0 to 4": set(np.unique(tumor).tolist()) <= {0, 1, 2, 3, 4},
    "tumor mask is not empty": bool(tumor_voxels.any()),
    "brain volume is plausible (900 to 2000 cc)": 900 <= brain_cc <= 2000,
    "at least 99% of the tumor is inside the brain mask": bool(tumor_voxels.any()) and (brain[tumor_voxels] > 0).mean() >= 0.99,
}
RESULTS["checks"] = {name: bool(value) for name, value in checks.items()}
RESULTS["brain_volume_cc"] = round(brain_cc, 1)
RESULTS["tumor_volumes_cc"] = volumes_cc(tumor, tumor_image)

for name, passed in checks.items():
    print("PASS" if passed else "FAIL", name)
print(f"\nBrain volume: {brain_cc:.0f} cc")
print("Region volumes (cc):", RESULTS["tumor_volumes_cc"])

## 8. Look at the overlays

Colors: yellow is NETC, green is SNFH, red is ET, and blue is RC. The cyan line is the brain mask.

Things to look for:

- **ET** should follow the bright enhancing rim on T1 post-contrast.
- **SNFH** should follow the bright edema on FLAIR.
- The **brain mask** should follow the brain, not the skull.
- **RC** should be absent or tiny, because this looks like a pre-operative case.

FLAIR is resampled onto the T1 post-contrast grid using the scanner's coordinates, for display only. Small misalignment here can come from patient motion.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

COLORS = ListedColormap([(0, 0, 0, 0), "#ffd400", "#2ecc40", "#ff2d2d", "#2f7fff"])


def to_ras(image, volume=None):
    """Return an array and spacing in RAS order for display."""
    source = image if volume is None else nib.Nifti1Image(volume, image.affine)
    canonical = nib.as_closest_canonical(source)
    return np.asanyarray(canonical.dataobj), canonical.header.get_zooms()[:3]


def take(volume, axis, index):
    return np.rot90(np.take(volume, index, axis=axis))


def show_overlays(mask, title, save_as=None):
    flair = resample_from_to(nib.load(NIFTI / "fl.nii.gz"), reference, order=1)
    tc_data, zooms = to_ras(reference)
    fl_data, _ = to_ras(flair)
    mask_data, _ = to_ras(reference, mask.astype(np.int16))
    brain_data, _ = to_ras(reference, (brain > 0).astype(np.int16))
    tumor_data = np.isin(mask_data, list(LABELS))
    center = [int(round(c)) for c in ndimage.center_of_mass(tumor_data)] if tumor_data.any() else [s // 2 for s in mask_data.shape]
    planes = [(2, "axial", (zooms[1], zooms[0])), (1, "coronal", (zooms[2], zooms[0])), (0, "sagittal", (zooms[2], zooms[1]))]

    figure, axes = plt.subplots(3, 4, figsize=(14, 10.5))
    for row, (axis, plane, (height, width)) in enumerate(planes):
        for column, (data, name, overlay) in enumerate(
            [(tc_data, "T1 post", False), (tc_data, "T1 post + mask", True), (fl_data, "FLAIR", False), (fl_data, "FLAIR + mask", True)]
        ):
            ax = axes[row, column]
            slice_data = take(data, axis, center[axis])
            ax.imshow(slice_data, cmap="gray", aspect=height / width, vmin=0, vmax=np.percentile(data[data > 0], 99.5))
            if overlay:
                ax.imshow(take(mask_data, axis, center[axis]), cmap=COLORS, vmin=0, vmax=4, alpha=0.5, aspect=height / width, interpolation="nearest")
                ax.contour(take(brain_data, axis, center[axis]), levels=[0.5], colors="cyan", linewidths=0.6)
            ax.set_title(f"{plane}: {name}", fontsize=10)
            ax.axis("off")
    figure.suptitle(title)
    figure.tight_layout()
    if save_as:
        figure.savefig(save_as, dpi=110)
    plt.show()


show_overlays(tumor, "UPENN-GBM-00622 from DICOM", OUT / "dicom" / "overlay.png")

## 9. Compare DICOM and NIfTI inputs

Both should give nearly the same mask. Registration can vary slightly between runs, so small differences are normal. Dice well below 0.95 for a large region needs a closer look.

In [ ]:
def dice(a, b):
    total = a.sum() + b.sum()
    return round(float(2 * (a & b).sum() / total), 4) if total else None


if RUN_NIFTI_COMPARISON:
    segment("nifti", {name: NIFTI / f"{name}.nii.gz" for name in SEQUENCES})
    _, nifti_mask = load_mask("nifti")
    RESULTS["dicom_vs_nifti_dice"] = {name: dice(np.isin(tumor, labels), np.isin(nifti_mask, labels)) for name, labels in REGIONS.items()}
    print("Dice between runs:", RESULTS["dicom_vs_nifti_dice"])

## 10. Try `--postprocess`

The tumor (labels 1, 2, and 3 together) and the resection cavity should each end up with at most one connected component.

In [ ]:
def components(mask, labels):
    return int(ndimage.label(np.isin(mask, labels))[1])


if RUN_POSTPROCESS:
    segment("postprocess", DICOM_INPUTS, "--postprocess")
    _, post_mask = load_mask("postprocess")
    RESULTS["components"] = {
        "tumor_before": components(tumor, [1, 2, 3]), "tumor_after": components(post_mask, [1, 2, 3]),
        "rc_before": components(tumor, [4]), "rc_after": components(post_mask, [4]),
    }
    RESULTS["postprocess_single_component"] = RESULTS["components"]["tumor_after"] <= 1 and RESULTS["components"]["rc_after"] <= 1
    print(RESULTS["components"])

## 11. Check that bad DICOM folders give clear errors

Each run below should fail on purpose, with a clear message.

In [ ]:
def expect_error(name, folder, message):
    inputs = dict(DICOM_INPUTS, t1=folder)
    code, output = run(
        "mist_glioma", "--t1", inputs["t1"], "--t2", inputs["t2"], "--tc", inputs["tc"], "--fl", inputs["fl"],
        "--output", OUT / "should_not_exist", "--device", "cuda", check=False,
    )
    passed = code != 0 and message in output
    RESULTS.setdefault("error_handling", {})[name] = passed
    print("PASS" if passed else "FAIL", name)


BAD = ROOT_PATH / "bad_dicom"
shutil.rmtree(BAD, ignore_errors=True)

two_series = BAD / "two_series"
shutil.copytree(CASE / "t1", two_series)
for path in (CASE / "fl").glob("*.dcm"):
    shutil.copy(path, two_series / f"fl_{path.name}")
expect_error("two series in one folder", two_series, "Found 2 DICOM series")

missing_slice = BAD / "missing_slice"
shutil.copytree(CASE / "t1", missing_slice)
sorted(missing_slice.glob("*.dcm"), key=lambda p: int(p.stem.split("-")[-1]))[96].unlink()
expect_error("missing slice", missing_slice, "missing, repeated, or unevenly spaced")

BAD.joinpath("empty").mkdir()
expect_error("empty folder", BAD / "empty", "No DICOM series")

## 12. Summary

Copy this output and paste it back into the chat. The second cell downloads the masks and the overlay image.

In [ ]:
import json

passed = all(RESULTS.get("checks", {"missing": False}).values()) and all(RESULTS.get("error_handling", {"missing": False}).values()) and RESULTS.get("unit_tests_passed", False)
print("OVERALL:", "ALL AUTOMATIC CHECKS PASSED (the overlays still need your eye)" if passed else "SOMETHING FAILED, SEE BELOW")
print(json.dumps(RESULTS, indent=2, default=str))

In [ ]:
from google.colab import files

archive = shutil.make_archive(str(ROOT_PATH / "mist_glioma_test_outputs"), "zip", OUT)
files.download(archive)